# Adversarial Arena - Unsloth Colab Training

This notebook trains on `data/trajectories.json` with high-reward filtering and saves a lightweight policy to `trained_model/`.

In [1]:
!pip -q install unsloth transformers datasets trl accelerate matplotlib

In [2]:
import os, json, subprocess, sys
import matplotlib.pyplot as plt
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

ROOT = '/content'
REPO_DIR = os.path.join(ROOT, 'SRS')
REPO_URL = os.environ.get('SRS_REPO_URL', '').strip()  # optional: set this if you want auto-clone

# Auto-clone project if requested and not present.
if REPO_URL and not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])

PROJECT_ROOT = REPO_DIR if os.path.exists(REPO_DIR) else ROOT
os.chdir(PROJECT_ROOT)

os.makedirs('trained_model', exist_ok=True)
os.makedirs('plots', exist_ok=True)
os.makedirs('training/artifacts', exist_ok=True)
os.makedirs('data', exist_ok=True)

DATA_PATH = 'data/trajectories.json'

# Auto-generate trajectories if missing.
if not os.path.exists(DATA_PATH):
    candidate = 'training/collect_trajectories.py'
    if os.path.exists(candidate):
        subprocess.check_call([sys.executable, candidate])

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        'Missing data/trajectories.json. Either set SRS_REPO_URL for auto-clone or place trajectories file in data/.'
    )

with open(DATA_PATH, 'r') as f:
    trajectories = json.load(f)

high = [x for x in trajectories if float(x['reward']) > 0.5]
train_rows = high if len(high) >= 100 else trajectories

# Keep diversity by capping repeated (opponent_action, action) pairs.
pair_cap = {}
diverse_rows = []
for row in train_rows:
    key = (row.get('opponent_action', 'na'), row.get('action', 'balanced'))
    pair_cap[key] = pair_cap.get(key, 0)
    if pair_cap[key] < 220:
        diverse_rows.append(row)
        pair_cap[key] += 1


def to_text(row):
    likely = max(row['belief'], key=row['belief'].get)
    likely_p = round(100 * row['belief'][likely], 1)
    prompt = (
        f"State: {json.dumps(row['state'])}\n"
        f"Belief: {json.dumps(row['belief'])}\n"
        f"History: {json.dumps(row['history'])}\n"
        f"Opponent most likely action: {likely} ({likely_p}%).\n"
        "Choose best counter and avoid repeated same action unless optimal.\n"
        "Output strict JSON only."
    )
    out = json.dumps({'action': row['action']})
    return {'text': prompt + '\n' + out}


dataset = Dataset.from_list([to_text(x) for x in diverse_rows])
print('project_root', PROJECT_ROOT)
print('dataset_size', len(dataset), 'filtered', len(high), 'diverse', len(diverse_rows))

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
model_name = 'unsloth/mistral-7b-instruct-v0.2-bnb-4bit'
model, tokenizer = FastLanguageModel.from_pretrained(model_name=model_name, max_seq_length=1024, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=1024,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        max_steps=80,
        learning_rate=2e-4,
        logging_steps=10,
        output_dir='training/artifacts/unsloth_ckpt',
        report_to='none',
    ),
)
result = trainer.train()
model.save_pretrained('trained_model/unsloth_adapter')
tokenizer.save_pretrained('trained_model/unsloth_adapter')
result.metrics

In [ ]:
rewards = [float(x['reward']) for x in trajectories]
window = 20
ma = [sum(rewards[max(0,i-window+1):i+1]) / min(window, i+1) for i in range(len(rewards))]
plt.figure(figsize=(8,4))
plt.plot(rewards, alpha=0.25, label='step reward')
plt.plot(ma, label='moving average')
plt.xlabel('Step')
plt.ylabel('Reward')
plt.title('Reward vs Step')
plt.legend()
plt.tight_layout()
plt.savefig('plots/reward_curve.png', dpi=150)

with open('training/artifacts/notebook_summary.json', 'w') as f:
    json.dump({'dataset_size': len(dataset), 'filtered_samples': len(high), 'model': model_name}, f, indent=2)
print('saved trained_model/ and plots/reward_curve.png')